# 02 — Feature engineering и сборка ml_training_dataset

Цель:
- построить очищенный датасет `ml_training_dataset.csv` из `initial_dataset.csv`,
- добавить базовые и производные признаки,
- получить матрицу X, таргет y и мета-информацию о фичах для моделей.

In [1]:
import os
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.data_prep.build_ml_dataset import (
    load_initial_dataset,
    build_ml_dataset,
    save_ml_dataset,
)
from src.data_prep.feature_engineering import (
    load_ml_dataset,
    prepare_features,
)
from src.utils.config import INITIAL_DATASET_PATH, ML_DATASET_PATH

In [2]:
# 1) Загружаем сырой датасет
df_raw = load_initial_dataset()
print("Raw shape:", df_raw.shape)

# 2) Строим очищенный ml-датафрейм
df_ml = build_ml_dataset(df_raw)
print("ML df shape:", df_ml.shape)

# 3) Сохраняем его
save_ml_dataset(df_ml)
print("Saved ml_training_dataset.csv to:", ML_DATASET_PATH)

Raw shape: (15312, 30)
ML df shape: (15307, 35)
Saved ml_training_dataset.csv to: /Users/rustamakhmedzianov/Documents/Projects/aero_nbo_uplift/data/processed/ml_training_dataset.csv


In [3]:
df_ml_loaded = load_ml_dataset()
df_ml_loaded.head()
df_ml_loaded.info()
sorted(df_ml_loaded.columns.tolist())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15307 entries, 0 to 15306
Data columns (total 35 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   client_id                 15307 non-null  int64  
 1   treatment                 15307 non-null  int64  
 2   offer_id                  15307 non-null  int64  
 3   treatment_date            15307 non-null  object 
 4   offer_type                15307 non-null  object 
 5   offer_category            15307 non-null  object 
 6   cost                      15307 non-null  float64
 7   offer_AOV                 15307 non-null  float64
 8   channel                   15307 non-null  object 
 9   conversion                15307 non-null  int64  
 10  recency_days              15307 non-null  int64  
 11  frequency_90d             15307 non-null  int64  
 12  monetary_90d              15307 non-null  float64
 13  avg_order_value_lifetime  15307 non-null  float64
 14  total_

['age',
 'avg_discount_percent_90d',
 'avg_order_value_lifetime',
 'category_affinity_top1',
 'channel',
 'city_tier',
 'client_id',
 'conversion',
 'cost',
 'days_since_last_promo',
 'discounts_used_90d',
 'email_open_rate_30d',
 'favorite_category',
 'frequency_90d',
 'gender',
 'is_mobile_user',
 'monetary_90d',
 'offer_AOV',
 'offer_category',
 'offer_id',
 'offer_type',
 'price_segment',
 'push_enabled',
 'recency_days',
 'time_afternoon',
 'time_evening',
 'time_morning',
 'time_night',
 'total_orders_lifetime',
 'treatment',
 'treatment_date',
 'treatment_dow',
 'treatment_hour',
 'treatment_month',
 'visited_category_14d']

In [4]:
X, y, ids, meta = prepare_features(df_ml_loaded)

X.shape, y.shape
y.value_counts(normalize=True)
meta
X.head()
ids.head()

,client_id,offer_id
0,17850,0
1,17850,2
2,17850,2
3,17850,1
4,13047,1


In [5]:
import json

feature_meta_path = PROJECT_ROOT / "data" / "processed" / "feature_meta.json"
feature_meta_path.parent.mkdir(parents=True, exist_ok=True)

with open(feature_meta_path, "w", encoding="utf-8") as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)

feature_meta_path

PosixPath('/Users/rustamakhmedzianov/Documents/Projects/aero_nbo_uplift/data/processed/feature_meta.json')

## Итог

1. Исходный датасет `initial_dataset.csv` преобразован в очищенный `ml_training_dataset.csv`
   с удалением аномалий, утечек (`revenue_14d`) и добавлением временных фич.
2. Функция `prepare_features`:
   - добавляет derived-фичи (`discount_share`, `lf_check_to_offer_ratio`,
     бинарные признаки категориальных совпадений),
   - приводит типы,
   - формирует:
     - `X` — матрица признаков,
     - `y` — таргет `conversion`,
     - `ids` — `client_id`, `offer_id`,
     - `meta` — словарь со списками числовых и категориальных фич.
3. На следующих шагах:
   - `03_rule_based.ipynb` — baseline на основе правил,
   - `04_uplift_training.ipynb` — обучение T-learner uplift-моделей,
   - `05_uplift_vs_rule_based.ipynb` — сравнение по uplift и экономике.